# Cedar City Wildfire Prediction — Team Notebook
CS 4150 • Cedar City, Utah • 50-mile radius

Run the cells from top to bottom, or choose **Runtime → Run all**. This notebook includes the Python files and public source data. You do not need to upload CSV files separately.

The three agents collect data, estimate risk, and check alerts. The model estimates a recorded fire somewhere in the area, not an exact ignition location. Three-day estimates are experimental; this is not an official warning system.

**Editing:** change a Python file in its code cell, run that cell again, then rerun training or the demo. Runtime files are temporary, but the code and data embedded in this saved notebook remain available. Email stays disabled; no passwords are included.


## 1. Install the libraries
Colab usually has these libraries already. This cell installs any missing packages. We retrain inside Colab so the model matches its installed library versions.

In [ ]:
%pip -q install numpy pandas scikit-learn scipy joblib requests


## 2. Restore the included data
Run this cell once per new runtime. It restores the original public agency downloads, source-named CSV folders, project guides, and tests. The long encoded text is a compressed data copy; edit the readable Python cells below.

In [ ]:
# Restore only the public project data bundled with this notebook.
import base64
import io
import os
import zipfile
from pathlib import Path

project_folder = Path('/content/cedar_fire') if Path('/content').exists() else Path.cwd() / 'cedar_fire'
project_folder.mkdir(parents=True, exist_ok=True)
os.chdir(project_folder)
data_copy = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(data_copy))) as archive:
    for item in archive.infolist():
        destination = (project_folder / item.filename).resolve()
        if not destination.is_relative_to(project_folder.resolve()):
            raise ValueError('Invalid file path in data archive')
    archive.extractall(project_folder)
print('Project folder:', project_folder)
print('Data ready. Next, run the Python file cells below.')


### The three agents — `core.py`
DataAgent collects information, PredictionAgent loads the trained model, and AlertAgent saves alerts.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile core.py
"""CS 4150 prototype: real historical labels, chronological evaluation, three agents."""

import datetime as dt
import hashlib
import json
import math
import os
import subprocess
import threading
import time
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import joblib

# Shared settings: folder paths, Cedar City coordinates, distance, and local time.
ROOT = Path(__file__).resolve().parent
CENTER = (37.6837408, -113.095454)
RADIUS = 50
TZ = ZoneInfo("America/Denver")
# Official public data services. No paid AI service is used.
USFS = "https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_FireOccurrenceCurrentEdition_01/MapServer/0"
NIFC = "https://services3.arcgis.com/T4QMspbfLg3qTGWY/arcgis/rest/services/WFIGS_Incident_Locations_Current/FeatureServer/0"
POWER = "https://power.larc.nasa.gov/api/temporal/daily/point"
FEATURES = [
    "sin_day",
    "cos_day",
    "temperature_max_7d",
    "humidity_7d",
    "wind_7d",
    "rain_7d",
]


def now():
    """Return the current UTC time for saved records."""
    return dt.datetime.now(dt.timezone.utc)


def distance(latitude, longitude):
    """Calculate miles from Cedar City using latitude and longitude."""
    # The haversine formula measures distance along the Earth in miles.
    center_latitude, point_latitude = map(math.radians, [CENTER[0], latitude])
    longitude_change = math.radians(longitude - CENTER[1])
    half_chord = (
        math.sin((point_latitude - center_latitude) / 2) ** 2
        + math.cos(center_latitude)
        * math.cos(point_latitude)
        * math.sin(longitude_change / 2) ** 2
    )
    return 3958.7613 * 2 * math.asin(math.sqrt(min(1, half_chord)))


def fetch(url, parameters=None):
    """Download public JSON data. Report failures instead of inventing data."""
    from urllib.parse import urlencode

    if parameters:
        url += "?" + urlencode(parameters)
    # Use the computer's trusted HTTPS certificates; never disable TLS verification.
    response_text = subprocess.check_output(
        [
            "curl",
            "--fail",
            "-L",
            "--max-time",
            "35",
            "--retry",
            "1",
            "-sS",
            "-H",
            "User-Agent: CedarFire-CS4150/0.1 (local academic prototype)",
            url,
        ],
        stderr=subprocess.PIPE,
        text=True,
        timeout=80,
    )
    response_data = json.loads(response_text)
    if isinstance(response_data, dict) and "error" in response_data:
        raise ValueError(str(response_data["error"])[:200])
    return response_data


def weather_frame(weather_response):
    """Convert NASA readings into a date-sorted table and mark missing values."""
    # NASA uses large negative numbers for missing readings. Convert them to NaN.
    weather_table = pd.DataFrame(weather_response["properties"]["parameter"])
    weather_table.index = pd.to_datetime(weather_table.index, format="%Y%m%d")
    return weather_table.astype(float).mask(lambda values: values <= -900).sort_index()


def features(target_dates, weather_table):
    """Build the six model inputs using only earlier weather and season."""
    # For target day T, use only weather from T-14 through T-8.
    # Requiring all seven days prevents missing readings from silently changing inputs.
    last_date = max(weather_table.index.max(), max(target_dates))
    daily_dates = pd.date_range(weather_table.index.min(), last_date)
    daily_weather_table = weather_table.reindex(daily_dates)
    seven_day_averages = daily_weather_table.rolling(7, min_periods=7).mean()
    past_weather = seven_day_averages.shift(8)
    # Sine/cosine represent the season without a sudden jump at New Year.
    input_rows = []
    for target_date in target_dates:
        day_of_year = target_date.day_of_year
        daily_weather = (
            past_weather.loc[target_date]
            if target_date in past_weather.index
            else pd.Series(dtype=float)
        )
        input_rows.append(
            [
                math.sin(2 * math.pi * day_of_year / 365.25),
                math.cos(2 * math.pi * day_of_year / 365.25),
            ]
            + [
                daily_weather.get(weather_column, np.nan)
                for weather_column in ["T2M_MAX", "RH2M", "WS10M", "PRECTOTCORR"]
            ]
        )
    return np.asarray(input_rows)


def score(model, input_values):
    """Get a model score that can be adjusted into a probability."""
    # SVM/logistic models provide a raw score; MLP provides probabilities.
    # Both paths give one score per row to the probability-adjustment model.
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(input_values)).reshape(-1, 1)
    probabilities = np.clip(model.predict_proba(input_values)[:, 1], 1e-06, 1 - 1e-06)
    return np.log(probabilities / (1 - probabilities)).reshape(-1, 1)


class DataAgent:
    """Agent 1: collect and check official data."""

    name = "01 · Data scout"

    def collect(self, target_date, forecast_days=1):
        """Gather weather inputs, official fire reports, and weather warnings."""
        if forecast_days not in (1, 2, 3):
            raise ValueError("Forecast horizon must be 1, 2 or 3 days")
        result = {
            "agent": self.name,
            "collected_at": now().isoformat(),
            "sources": {},
            "incidents": [],
            "warnings": [],
            "weather": None,
            "forecast": None,
        }

        # One unavailable service should not stop us checking the other services.
        def source(name, get_data):
            """Record whether one source was available; allow other sources to continue."""
            try:
                source_data = get_data()
                result["sources"][name] = {
                    "status": "ok",
                    "retrieved_at": now().isoformat(),
                }
                return source_data
            except Exception as error:
                result["sources"][name] = {
                    "status": "unavailable",
                    "detail": str(error)[:160],
                }
                return None

        # Request enough older weather to build all three daily inputs at once.
        start_date = target_date - dt.timedelta(days=25)
        end_date = (
            target_date + dt.timedelta(days=forecast_days - 1) - dt.timedelta(days=8)
        )
        weather_response = source(
            "NASA POWER",
            lambda: fetch(
                POWER,
                dict(
                    parameters="T2M_MAX,RH2M,WS10M,PRECTOTCORR",
                    community="AG",
                    latitude=CENTER[0],
                    longitude=CENTER[1],
                    start=start_date.strftime("%Y%m%d"),
                    end=end_date.strftime("%Y%m%d"),
                    format="JSON",
                    **{"time-standard": "UTC"},
                ),
            ),
        )
        if weather_response:
            try:
                weather_table = weather_frame(weather_response)
                input_values = features(pd.DatetimeIndex([target_date]), weather_table)
                if not np.isfinite(input_values).all():
                    raise ValueError(
                        "Required lagged weather days are missing; prediction withheld"
                    )
                result["weather"] = {
                    "features": input_values[0].tolist(),
                    "window_start": (target_date - dt.timedelta(days=14)).isoformat(),
                    "window_end": (target_date - dt.timedelta(days=8)).isoformat(),
                    "units": {
                        "temperature": "C",
                        "humidity": "percent",
                        "wind": "m/s",
                        "rain": "mm/day",
                    },
                    "location": list(CENTER),
                }
                result["weather_horizon"] = {}
                # Each prediction day gets its own earlier seven-day weather window.
                for offset in range(forecast_days):
                    prediction_date = target_date + dt.timedelta(days=offset)
                    daily_inputs = features(
                        pd.DatetimeIndex([prediction_date]), weather_table
                    )
                    if np.isfinite(daily_inputs).all():
                        result["weather_horizon"][prediction_date.isoformat()] = {
                            **result["weather"],
                            "features": daily_inputs[0].tolist(),
                            "window_start": (
                                prediction_date - dt.timedelta(days=14)
                            ).isoformat(),
                            "window_end": (
                                prediction_date - dt.timedelta(days=8)
                            ).isoformat(),
                        }
            except Exception as error:
                result["sources"]["NASA POWER"] = {
                    "status": "incomplete",
                    "detail": str(error),
                }

        def incidents():
            """Read all result pages and keep wildfire points inside our radius."""
            local_fires = []
            # The fire service returns results in pages, not necessarily all at once.
            for offset in range(0, 20000, 2000):
                fire_response = fetch(
                    NIFC + "/query",
                    dict(
                        f="json",
                        where="IncidentTypeCategory = 'WF'",
                        geometry=f"{CENTER[1]},{CENTER[0]}",
                        geometryType="esriGeometryPoint",
                        inSR=4326,
                        distance=RADIUS * 1609.344,
                        units="esriSRUnit_Meter",
                        spatialRel="esriSpatialRelIntersects",
                        outFields="*",
                        outSR=4326,
                        returnGeometry="true",
                        resultOffset=offset,
                        resultRecordCount=2000,
                        orderByFields="OBJECTID",
                    ),
                )
                for fire_record in fire_response.get("features", []):
                    fire_info = fire_record["attributes"]
                    fire_location = fire_record.get("geometry") or {}
                    latitude = fire_location.get("y")
                    longitude = fire_location.get("x")
                    # A nearby-state report is not automatically inside our circle.
                    if (
                        latitude is None
                        or longitude is None
                        or distance(latitude, longitude) > RADIUS
                    ):
                        continue
                    local_fires.append(
                        {
                            "id": fire_info.get("UniqueFireIdentifier")
                            or fire_info.get("IrwinID")
                            or str(fire_info.get("OBJECTID")),
                            "name": fire_info.get("IncidentName", "Unnamed incident"),
                            "lat": latitude,
                            "lon": longitude,
                            "distance_miles": round(distance(latitude, longitude), 1),
                            "acres": fire_info.get("DailyAcres")
                            or fire_info.get("IncidentSize"),
                            "contained": fire_info.get("PercentContained"),
                            "out_at": fire_info.get("FireOutDateTime"),
                            "modified": fire_info.get("ModifiedOnDateTime"),
                            "discovered": fire_info.get("FireDiscoveryDateTime"),
                            "source": NIFC,
                        }
                    )
                if not fire_response.get("exceededTransferLimit"):
                    return local_fires
            raise ValueError("Incomplete incident pagination")

        result["incidents"] = source("NIFC WFIGS", incidents) or []
        weather_alerts = source(
            "NWS alerts",
            lambda: fetch(
                "https://api.weather.gov/alerts/active",
                {"area": "UT,NV,AZ", "event": "Red Flag Warning,Fire Weather Watch"},
            ),
        )
        if weather_alerts:
            for alert_record in weather_alerts.get("features", []):
                alert_info = alert_record["properties"]
                geometry = alert_record.get("geometry")
                scope = "state context — exact overlap unverified"
                if geometry:
                    scope = (
                        "within radius"
                        if geometry_intersects(geometry)
                        else "outside radius"
                    )
                # Some warnings name forecast zones instead of providing a shape.
                # Check those zone shapes before treating the warning as local.
                elif alert_info.get("affectedZones"):
                    verified = False
                    for zone in alert_info["affectedZones"]:
                        try:
                            zone_response = fetch(zone)
                            zone_shape = zone_response.get("geometry")
                            if zone_shape and geometry_intersects(zone_shape):
                                scope = "within radius"
                                verified = True
                                break
                            if not zone_shape:
                                raise ValueError("Zone geometry missing")
                        except Exception:
                            break
                    else:
                        verified = True
                        scope = "outside radius"
                if scope == "outside radius":
                    continue
                result["warnings"].append(
                    {
                        "id": alert_record.get("id", alert_info.get("id")),
                        "event": alert_info.get("event"),
                        "headline": alert_info.get("headline"),
                        "description": alert_info.get("description"),
                        "expires": alert_info.get("expires"),
                        "scope": scope,
                        "area": alert_info.get("areaDesc"),
                        "source": alert_info.get("@id") or alert_record.get("id"),
                    }
                )
        location_info = source(
            "NWS forecast",
            lambda: fetch(
                f"https://api.weather.gov/points/{CENTER[0]:.4f},{CENTER[1]:.4f}"
            ),
        )
        if location_info:
            hourly_forecast = source(
                "NWS hourly forecast",
                lambda: fetch(location_info["properties"]["forecastHourly"]),
            )
            if hourly_forecast:
                result["forecast"] = {
                    "updated": hourly_forecast["properties"].get("updateTime"),
                    "periods": hourly_forecast["properties"].get("periods", [])[:24],
                }
        return result


def geometry_intersects(area_shape):
    """Approximate whether a warning area overlaps the circle around Cedar City."""
    # Approximate warning-area overlap on a local map, with a one-mile margin.
    # This is for a prototype, not exact fire-department jurisdiction mapping.
    polygons = (
        [area_shape["coordinates"]]
        if area_shape["type"] == "Polygon"
        else area_shape["coordinates"] if area_shape["type"] == "MultiPolygon" else []
    )
    for polygon in polygons:
        outer_edge = polygon[0]
        map_points = [
            (
                (longitude - CENTER[1]) * 69.172 * math.cos(math.radians(CENTER[0])),
                (latitude - CENTER[0]) * 69.0,
            )
            for longitude, latitude, *_ in outer_edge
        ]
        inside = False
        for (start_x, start_y), (end_x, end_y) in zip(
            map_points, map_points[1:] + map_points[:1]
        ):
            if (start_y > 0) != (end_y > 0) and 0 < (end_x - start_x) * -start_y / (
                end_y - start_y
            ) + start_x:
                inside = not inside
            segment_length_squared = (end_x - start_x) ** 2 + (end_y - start_y) ** 2
            nearest_fraction = (
                max(
                    0,
                    min(
                        1,
                        -(start_x * (end_x - start_x) + start_y * (end_y - start_y))
                        / segment_length_squared,
                    ),
                )
                if segment_length_squared
                else 0
            )
            if (
                math.hypot(
                    start_x + nearest_fraction * (end_x - start_x),
                    start_y + nearest_fraction * (end_y - start_y),
                )
                <= RADIUS + 1
            ):
                return True
        if inside:
            return True
    return False


class PredictionAgent:
    """Agent 2: estimate the chance of a reported wildfire."""

    name = "02 · Risk analyst"

    def __init__(self):
        # Load only our own trained file; joblib files from strangers are unsafe.
        self.bundle = joblib.load(ROOT / "artifacts/model.joblib")

    def predict(self, collected_data, target_date):
        """Use the saved classifier and probability adjustment for one target date."""
        if not collected_data.get("weather"):
            return {
                "agent": self.name,
                "status": "withheld",
                "reason": "Required NASA weather inputs unavailable. No estimate is substituted.",
            }
        # Use the same feature order and saved scaling as during training.
        saved_model = self.bundle
        input_values = np.array([collected_data["weather"]["features"]])
        # Adjust the raw score using the separate probability-calibration years.
        fire_probability = float(
            saved_model["calibrator"].predict_proba(
                score(saved_model["model"], input_values)
            )[0, 1]
        )
        return {
            "agent": self.name,
            "status": "estimated",
            "probability": fire_probability,
            "threshold": 0.5,
            "level": (
                "Elevated" if fire_probability >= 0.5 else "Below prototype threshold"
            ),
            "target_date": target_date.isoformat(),
            "model": saved_model["name"],
            "definition": "At least one recorded wildfire discovery anywhere inside the 50-mile circle on the next America/Denver calendar day. This is a regional historical-risk estimate, not an ignition location or spread forecast.",
            "weather_window": collected_data["weather"],
            "validation": saved_model["report"],
        }


class AlertAgent:
    """Agent 3: save alerts and avoid repeat notifications."""

    name = "03 · Alert coordinator"

    def __init__(self):
        self.path = ROOT / "data/alerts.json"
        self.lock = threading.Lock()
        if not self.path.exists():
            self.path.write_text("[]")

    def history(self):
        """Read previously saved alerts so they can be listed or checked for duplicates."""
        return json.loads(self.path.read_text())

    def emit(self, kind, key, title, body, test=False):
        """Save one new alert unless the same event has already been recorded."""
        # The same event receives the same ID, preventing duplicate notifications.
        alert_id = hashlib.sha256((kind + key).encode()).hexdigest()[:20]
        with self.lock:
            saved_alerts = self.history()
            if any((saved_alert["id"] == alert_id for saved_alert in saved_alerts)):
                return None
            new_alert = {
                "id": alert_id,
                "kind": kind,
                "title": title,
                "body": body,
                "created_at": now().isoformat(),
                "test": test,
                "delivery": "terminal",
                "department_delivery": "disabled",
                "acknowledged": False,
            }
            saved_alerts.insert(0, new_alert)
            self.path.write_text(json.dumps(saved_alerts[:500], indent=2))
            return new_alert

    def evaluate(self, data, prediction):
        """Check model risk, official weather warnings, and recent fire reports."""
        # Model estimates, official warnings, and reported fires are different evidence.
        risk_result = prediction
        if (
            risk_result.get("status") == "estimated"
            and risk_result["probability"] >= risk_result["threshold"]
        ):
            self.emit(
                "model-risk",
                risk_result["target_date"],
                "Prototype regional risk threshold reached",
                f"Model estimate: {risk_result['probability']:.1%} for at least one recorded discovery across the entire region on {risk_result['target_date']}. Verify with official sources. This is not a confirmed fire.",
            )
        for weather_warning in data.get("warnings", []):
            if (
                weather_warning["scope"] == "within radius"
                and weather_warning.get("expires")
                and (
                    dt.datetime.fromisoformat(
                        weather_warning["expires"].replace("Z", "+00:00")
                    )
                    > now()
                )
            ):
                self.emit(
                    "official-weather",
                    weather_warning["id"],
                    weather_warning["event"],
                    f"{weather_warning['headline']}\nExpires: {weather_warning['expires']}\nSource: {weather_warning['source']}\nFire weather warning does not confirm an ignition.",
                )
        # Only recently reported fires that are not out/fully contained trigger alerts.
        for fire in data.get("incidents", []):
            discovery_time = fire.get("discovered")
            is_recent = (
                discovery_time
                and now().timestamp() - discovery_time / 1000 < 48 * 3600
                and (discovery_time / 1000 <= now().timestamp())
            )
            if (
                is_recent
                and (not fire.get("out_at"))
                and ((fire.get("contained") or 0) < 100)
            ):
                self.emit(
                    "reported-incident",
                    fire["id"],
                    "Officially reported wildfire near Cedar City",
                    f"{fire['name']} · {fire['distance_miles']} miles from Cedar City · {fire['lat']}, {fire['lon']}\nReported discovery within 48 hours. Source: {fire['source']}",
                )

    def acknowledge(self, alert_id):
        """Mark a saved alert as read."""
        with self.lock:
            saved_alerts = self.history()
            for saved_alert in saved_alerts:
                if saved_alert["id"] == alert_id:
                    saved_alert["acknowledged"] = True
            self.path.write_text(json.dumps(saved_alerts, indent=2))


### Optional email — `email_alerts.py`
Simple Mail Transfer Protocol (SMTP) support. Delivery is disabled. Use private runtime secrets later; never put a password in this shared notebook.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile email_alerts.py
"""Email delivery for the terminal project. No web server or model is loaded here."""

import json
import os
import smtplib
import ssl
from email.message import EmailMessage
from core import ROOT, now, AlertAgent

# Read sender settings locally. Never print or commit the app password.
if (ROOT / ".env").exists():
    for line in (ROOT / ".env").read_text().splitlines():
        if line.strip() and (not line.lstrip().startswith("#")) and ("=" in line):
            setting_name, setting_value = line.split("=", 1)
            os.environ.setdefault(
                setting_name.strip(), setting_value.strip().strip('"').strip("'")
            )
RECIPIENT = os.getenv("ALERT_RECIPIENT", "")


def email_status():
    """Check local email settings without showing any password."""
    return {
        "recipient": RECIPIENT,
        "configured": bool(os.getenv("SMTP_USER") and os.getenv("SMTP_PASSWORD")),
        "enabled": os.getenv("EMAIL_ENABLED", "false").lower() == "true",
        "department_delivery": "disabled",
    }


def mail(alert):
    """Send one alert by email and save whether the provider accepted it."""
    alert_agent = AlertAgent()
    email_settings = email_status()
    if not email_settings["enabled"] or not email_settings["configured"]:
        return {
            "status": "not_sent",
            "reason": "Email credentials or EMAIL_ENABLED are not configured",
        }
    # Keep the recipient fixed to the project owner; departments are not contacted.
    message = EmailMessage()
    message["From"] = os.environ["SMTP_USER"]
    message["To"] = RECIPIENT
    message["Subject"] = (
        "[TEST] " if alert.get("test") else "[CEDAR FIRE PROTOTYPE] "
    ) + alert["title"]
    message["Message-ID"] = f"<{alert['id']}@cedar-fire.local>"
    message.set_content(
        alert["body"]
        + "\n\nAcademic prototype — not an official warning or emergency dispatch.\n50-mile radius of Cedar City, Utah. Verify official NWS/NIFC information.\nAlert ID: "
        + alert["id"]
    )
    # TLS encrypts the SMTP connection and verifies the mail server certificate.
    try:
        with smtplib.SMTP_SSL(
            os.getenv("SMTP_HOST", "smtp.gmail.com"),
            int(os.getenv("SMTP_PORT", "465")),
            context=ssl.create_default_context(),
            timeout=25,
        ) as email_server:
            email_server.login(os.environ["SMTP_USER"], os.environ["SMTP_PASSWORD"])
            email_server.send_message(message)
        result = {
            "status": "accepted_by_smtp",
            "at": now().isoformat(),
            "recipient": RECIPIENT,
        }
    except Exception as error:
        result = {
            "status": "failed",
            "reason": type(error).__name__ + " — check local sender configuration",
        }
    # Provider acceptance is recorded; it is not proof of inbox delivery.
    with alert_agent.lock:
        saved_alerts = alert_agent.history()
        for saved_alert in saved_alerts:
            if saved_alert["id"] == alert["id"]:
                saved_alert["email"] = result
        alert_agent.path.write_text(json.dumps(saved_alerts, indent=2))
    return result


### The next three days — `forecast.py`
Runs the daily model for three target dates. Internet is needed for current data.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile forecast.py
"""Show three daily regional risk estimates using the saved model and official data."""

import datetime as dt
import json
from core import ROOT, RADIUS, TZ, now, DataAgent, PredictionAgent, AlertAgent


def show_next_three_days(send_email=False):
    """Run the three agents and print one short result row for each day."""
    # Tomorrow is based on Cedar City time, not the computer server's timezone.
    first_date = dt.datetime.now(TZ).date() + dt.timedelta(days=1)
    print("CEDAR CITY, UTAH | WILDFIRE RISK", flush=True)
    print(f"Area: within {RADIUS} miles of Cedar City")
    print("Getting recent weather data...", flush=True)
    # Collect once, then reuse the readings for all three target dates.
    collected_data = DataAgent().collect(first_date, forecast_days=3)
    prediction_agent = PredictionAgent()
    alert_agent = AlertAgent()
    predictions = []
    old_alert_ids = {saved_alert["id"] for saved_alert in alert_agent.history()}
    print("\nDate          Chance of a fire report    Risk alert")
    print("--------------------------------------------------")
    # Predict each day separately; do not reuse one probability for all days.
    for day_number in range(3):
        target_date = first_date + dt.timedelta(days=day_number)
        daily_weather = collected_data.get("weather_horizon", {}).get(
            target_date.isoformat()
        )
        daily_data = {**collected_data, "weather": daily_weather}
        daily_prediction = prediction_agent.predict(daily_data, target_date)
        predictions.append(daily_prediction)
        # Missing inputs remain unavailable instead of being filled with fake readings.
        if daily_prediction["status"] == "estimated":
            chance_text = f"{daily_prediction['probability']:.1%}"
            alert_text = (
                "YES"
                if daily_prediction["probability"] >= daily_prediction["threshold"]
                else "NO"
            )
        else:
            chance_text = "Unavailable"
            alert_text = "Cannot decide"
        print(f"{target_date.isoformat()}    {chance_text:<26}{alert_text}")
        if daily_prediction["status"] == "estimated":
            alert_agent.evaluate(daily_data, daily_prediction)
    print("\nRisk alert level: 50% or higher.")
    print("Chance refers to any reported wildfire in the whole area, not one location.")
    print("Estimates use past weather and season; they are not official fire warnings.")
    if any(
        (saved_prediction["status"] != "estimated" for saved_prediction in predictions)
    ):
        print(
            "Some required weather readings could not be obtained. No replacement values were used."
        )
    # Email is optional. Send only alerts created during this run.
    if send_email:
        from email_alerts import mail

        new_alerts = [
            saved_alert
            for saved_alert in alert_agent.history()
            if saved_alert["id"] not in old_alert_ids
        ]
        for saved_alert in new_alerts:
            email_status = mail(saved_alert)["status"]
            print(
                "Email:",
                (
                    "accepted by provider"
                    if email_status == "accepted_by_smtp"
                    else "not sent — check sender setup"
                ),
            )
        if not new_alerts:
            print("Email: no new alerts to send.")
    else:
        print("Email: not sent.")
    # Save the latest results so the group can inspect them after the terminal run.
    (ROOT / "data/three_day_results.json").write_text(
        json.dumps(
            {
                "created_at": now().isoformat(),
                "radius_miles": RADIUS,
                "predictions": predictions,
            },
            indent=2,
        )
    )
    print()
    return predictions


### Historical demonstration — `demo.py`
Uses the included history for a repeatable class demonstration.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile demo.py
"""Plain-language terminal presentation of the three-agent wildfire project."""

import argparse
import datetime as dt
import json
import secrets
import pandas as pd
from core import (
    ROOT,
    RADIUS,
    TZ,
    DataAgent,
    PredictionAgent,
    AlertAgent,
    features,
    weather_frame,
)


def heading(title):
    """Print a heading only when the detailed explanation is requested."""
    print("\n" + title)
    print("-" * len(title))


def email_result(result):
    """Describe email success or failure without exposing login details."""
    status = result["status"]
    if status == "accepted_by_smtp":
        print(
            "  The email provider accepted the message. Check your inbox and spam folder."
        )
    elif status == "not_sent":
        print(
            "  Email was not sent. Connect a sending account with setup_email.py first."
        )
    else:
        print("  Email could not be sent. Please check the sending account settings.")


def run_details(argv=None):
    """Run an optional detailed historical or live demonstration."""
    argument_parser = argparse.ArgumentParser(
        description="Explain and run the Cedar City wildfire model in the terminal."
    )
    argument_parser.add_argument(
        "--live",
        action="store_true",
        help="Get current official data and estimate tomorrow’s risk.",
    )
    argument_parser.add_argument(
        "--date",
        default="2024-07-01",
        help="Historical test date, YYYY-MM-DD, in 2022–2024.",
    )
    argument_parser.add_argument(
        "--test-alert",
        action="store_true",
        help="Save a local demonstration alert. No email is sent.",
    )
    argument_parser.add_argument(
        "--send-test-email",
        action="store_true",
        help="Send a TEST email using your local sender setup.",
    )
    argument_parser.add_argument(
        "--send-email",
        action="store_true",
        help="Email new live alerts. Requires --live.",
    )
    argument_parser.add_argument(
        "--details", action="store_true", help="Show the longer project explanation."
    )
    options = argument_parser.parse_args(argv)
    if options.send_email and (not options.live):
        argument_parser.error(
            "--send-email requires --live. Past predictions are never emailed as current warnings."
        )
    prediction_agent = PredictionAgent()
    alert_agent = AlertAgent()
    report = prediction_agent.bundle["report"]
    fire_records = json.loads((ROOT / "data/fires.json").read_text())
    print("=" * 68)
    print("CEDAR CITY WILDFIRE PREDICTION | CS 4150 CLASS PROJECT")
    print("=" * 68)
    print(f"Place: Cedar City, Utah, and places within {RADIUS} miles.")
    print("Goal: estimate whether one or more wildfires will be reported")
    print("somewhere in this area on the next local calendar day.")
    print("The program uses three helpers, called agents.")
    print("This is a student prototype, not an official emergency warning.")
    if options.live:
        target_date = dt.datetime.now(TZ).date() + dt.timedelta(days=1)
        print("Mode: LIVE DATA. The date below is tomorrow in Cedar City.")
    else:
        try:
            target_date = dt.date.fromisoformat(options.date)
        except ValueError:
            argument_parser.error("Enter a date as YYYY-MM-DD.")
        if not dt.date(2022, 1, 1) <= target_date <= dt.date(2024, 12, 31):
            argument_parser.error(
                "Choose a historical test date from 2022 through 2024."
            )
        print("Mode: HISTORICAL CLASSROOM DEMO. This is NOT today’s forecast.")
        print("We make a prediction for a past test date, then check what happened.")
    heading("1. DATA AGENT — gets the information")
    print(
        f"  Past fire reports: {len(fire_records):,}, from 2000–2024, inside {RADIUS} miles."
    )
    print("  Fire source: US Forest Service (official reported fire history).")
    print("  Weather source: NASA POWER (temperature, air moisture, wind, rain).")
    print("  Live-only sources: National Weather Service warnings and forecasts;")
    print("                     National Interagency Fire Center fire reports.")
    if options.live:
        print(
            "  Getting current official information. This can take a little time...",
            flush=True,
        )
        collected_data = DataAgent().collect(target_date)
        for name, status in collected_data["sources"].items():
            text = "received" if status["status"] == "ok" else "missing or incomplete"
            print(f"  {name}: {text}.")
    else:
        weather_table = weather_frame(
            json.loads((ROOT / "data/weather_history.json").read_text())
        )
        input_values = features(pd.DatetimeIndex([target_date]), weather_table)
        collected_data = {
            "weather": {
                "features": input_values[0].tolist(),
                "window_start": (target_date - dt.timedelta(days=14)).isoformat(),
                "window_end": (target_date - dt.timedelta(days=8)).isoformat(),
            },
            "incidents": [],
            "warnings": [],
        }
        print(
            "  Reading the real downloaded files. No internet is needed for this demo."
        )
    if collected_data.get("weather"):
        weather_details = collected_data["weather"]
        weather_inputs = weather_details["features"]
        print(
            f"  Earlier weather used: {weather_details['window_start']} through {weather_details['window_end']} (UTC dates)."
        )
        print("  These are averages over seven days, measured near Cedar City:")
        print(f"    Daily high temperature: {weather_inputs[2]:.1f} °C")
        print(f"    Moisture in the air:   {weather_inputs[3]:.1f}% relative humidity")
        print(
            f"    Wind speed:            {weather_inputs[4]:.1f} meters per second ({weather_inputs[4] * 2.236936:.1f} mph)"
        )
        print(f"    Rainfall:              {weather_inputs[5]:.2f} mm per day")
        print("  The time of year is also an input. Future weather is not used.")
    else:
        print(
            "  Required weather readings are missing. We cannot give a risk estimate."
        )
    heading("2. PREDICTION AGENT — estimates the risk")
    result = prediction_agent.predict(collected_data, target_date)
    print(f"  Prediction date: {target_date} (Cedar City local calendar day).")
    if result["status"] == "estimated":
        print(
            f"  Estimated chance of one or more fire reports: {result['probability']:.1%}."
        )
        print("  This chance is for the WHOLE 50-mile area, not your home or one spot.")
        print("  Selected method:", result["model"])
        print("  This method learned a pattern from past fires, weather, and season.")
        print("  A higher percentage means greater estimated risk, not a certain fire.")
    else:
        print("  No estimate shown because required weather information is missing.")
        print("  Missing information does not mean there is no fire risk.")
    heading("3. ALERT AGENT — decides whether to notify you")
    print("  Our project rule: create a risk alert when the estimate is 50% or more.")
    print("  This is a chosen demo rule, not an official firefighting threshold.")
    alert_needed = (
        result.get("status") == "estimated"
        and result["probability"] >= result["threshold"]
    )
    if result["status"] != "estimated":
        print(
            "  Risk decision: no model-based alert decision; the estimate is unavailable."
        )
    elif alert_needed:
        print("  Risk decision: the estimate reaches the alert level.")
    else:
        print("  Risk decision: the estimate is below the alert level.")
        print("  A fire is still possible even when no risk alert is triggered.")
    if options.live:
        print(f"  Official fire reports found: {len(collected_data['incidents'])}.")
        print(
            f"  Relevant or nearby weather-warning records found: {len(collected_data['warnings'])}."
        )
        print(
            "  Only verified local weather warnings and recent qualifying fire reports"
        )
        print("  can create official-update alerts. A warning is not a confirmed fire.")
        existing = {item["id"] for item in alert_agent.history()}
        alert_agent.evaluate(collected_data, result)
        new_alerts = [
            item for item in alert_agent.history() if item["id"] not in existing
        ]
        print(f"  New alerts saved: {len(new_alerts)}. Repeated alerts are skipped.")
        for item in new_alerts:
            print("  ALERT:", item["title"])
            print(" ", item["body"])
            if options.send_email:
                from email_alerts import mail

                email_result(mail(item))
    else:
        print("  This is a past-date demo. No current warning is created or emailed.")
    if not options.live:
        heading("CHECK THE PAST RESULT — only after making the prediction")
        recorded_fire_count = sum(
            (
                pd.to_datetime(f["discovery_date"], unit="ms").date() == target_date
                for f in fire_records
            )
        )
        print(
            f"  Actual recorded fire reports on {target_date}: {recorded_fire_count}."
        )
        if result["status"] != "estimated":
            print("  No estimate was made, so this run cannot be compared.")
        elif alert_needed and recorded_fire_count == 0:
            print("  On this example, the rule would give a FALSE ALARM:")
            print("  it suggested an alert, but no fire was recorded that day.")
        elif alert_needed and recorded_fire_count > 0:
            print("  On this example, the rule correctly suggested an alert.")
        elif not alert_needed and recorded_fire_count > 0:
            print("  On this example, the rule MISSED a day with a recorded fire.")
        else:
            print("  On this example, no alert was suggested and no fire was recorded.")
        print(
            "  One example does not measure the whole model. See the full test below."
        )
    heading("HOW WE TRAINED AND CHECKED THE MODEL")
    print("  2000–2017: teach the models using earlier data.")
    print("  2018–2019: compare models and choose one.")
    print("  2020–2021: adjust the probability estimates.")
    print("  2022–2024: test on later data not used to teach or choose the model.")
    print("  Models compared (smaller prediction-error score is better):")
    for name, value in report["selection_brier"].items():
        print(f"    {name}: {value:.3f}")
    print("  The simple sigmoid model was chosen because it had the lowest")
    print("  comparison error. A more complex neural network did not win here.")
    print("  Class topics: intelligent agents, supervised learning, sigmoid,")
    print("  neural networks with backpropagation, RBF SVM, and separate test data.")
    heading("RESULTS ON ALL FINAL TEST DAYS — 2022–2024")
    correct_no_alert_days, false_alarm_days = report["confusion_matrix_at_0_5"][0]
    missed_fire_days, correct_alert_days = report["confusion_matrix_at_0_5"][1]
    print(f"  Days tested: {report['test_days']:,}.")
    print(f"  Correct alerts on days with a recorded fire: {correct_alert_days}.")
    print(f"  False alarms on days with no recorded fire: {false_alarm_days}.")
    print(f"  Missed days with a recorded fire: {missed_fire_days}.")
    print(
        f"  Correct no-alert decisions on days with no recorded fire: {correct_no_alert_days}."
    )
    print(
        f"  Out of 100 days that triggered an alert, about {report['precision_at_0_5'] * 100:.0f}"
    )
    print("  had a recorded fire. This is called precision.")
    print(
        f"  Out of 100 days with a recorded fire, the rule caught about {report['recall_at_0_5'] * 100:.0f}."
    )
    print("  This is called recall. These are past results, not future guarantees.")
    heading("NOTIFICATIONS AND SAVED FILES")
    from email_alerts import email_status

    status = email_status()
    ready = status["enabled"] and status["configured"]
    print("  Your alert email:", status["recipient"])
    print(
        "  Email setup:",
        "ready" if ready else "not ready — a sending account must be connected.",
    )
    print(
        "  Email on this run:",
        (
            "requested for new live alerts"
            if options.send_email
            else "off (unless you explicitly request a test email)."
        ),
    )
    print("  Fire-department notifications: off.")
    print("  CSV folders: data/US Forest Service, data/NASA, data/Model Data.")
    print("  Detailed model scores: artifacts/metrics.json.")
    print("  Saved local alerts: data/alerts.json.")
    if options.test_alert or options.send_test_email:
        item = alert_agent.emit(
            "test",
            secrets.token_hex(8),
            "Test alert — no wildfire reported",
            "This is a classroom test. It does not report a real fire.",
            True,
        )
        print("  TEST ONLY: a demonstration alert was saved.")
        if options.send_test_email:
            from email_alerts import mail

            email_result(mail(item))
    heading("WHAT THIS PROJECT CANNOT TELL YOU")
    print("  It cannot say exactly where a fire will start or how it will spread.")
    print("  Weather comes from one area; terrain, dry plants, lightning, and")
    print("  human activity are not yet included. Some fires may not be reported.")
    print("  Do not use this student model to make emergency-response decisions.")
    print("\nRun finished. No webpage was started.\n")
    return result


def main(argv=None):
    """Read command-line options and run the requested terminal mode."""
    import contextlib
    import io
    import sys

    arguments = list(sys.argv[1:] if argv is None else argv)
    if any((flag in arguments for flag in ("--details", "--help", "-h"))):
        return run_details(arguments)
    if "--date" not in arguments and (
        not any((flag in arguments for flag in ("--test-alert", "--send-test-email")))
    ):
        from forecast import show_next_three_days

        return show_next_three_days(send_email="--send-email" in arguments)
    live = "--live" in arguments
    print("CEDAR CITY WILDFIRE PREDICTION", flush=True)
    print(f"Area: Cedar City, Utah | {RADIUS}-mile radius")
    print(
        "Mode: "
        + (
            "Live data — checking tomorrow"
            if live
            else "Historical demo — not a current warning"
        ),
        flush=True,
    )
    if live:
        print("Getting weather and fire reports...", flush=True)
    output = io.StringIO()
    with contextlib.redirect_stdout(output):
        result = run_details(arguments)
    if result["status"] == "estimated":
        print("Date:", result["target_date"])
        print(
            f"Estimated chance of a fire report in the area: {result['probability']:.1%}"
        )
        print(
            "Risk alert: "
            + (
                "YES — estimate is 50% or higher"
                if result["probability"] >= result["threshold"]
                else "NO — estimate is below 50%"
            )
        )
    else:
        print("Risk estimate: unavailable — required weather readings are missing.")
        print("Risk alert: cannot decide without the missing readings.")
    for line in output.getvalue().splitlines():
        text = line.strip()
        if text.startswith(
            (
                "Actual recorded fire reports",
                "Official fire reports found:",
                "New alerts saved:",
                "TEST ONLY:",
                "ALERT:",
            )
        ):
            print(text)
        elif text.startswith("On this example, the rule would give a FALSE ALARM"):
            print("Historical result: false alarm — no fire was recorded that day.")
        elif text.startswith("On this example, the rule correctly suggested"):
            print("Historical result: alert matched a day with a recorded fire.")
        elif text.startswith("On this example, the rule MISSED"):
            print("Historical result: missed a day with a recorded fire.")
        elif text.startswith("On this example, no alert was suggested"):
            print("Historical result: no alert and no recorded fire.")
        elif text.startswith(
            (
                "The email provider accepted",
                "Email was not sent.",
                "Email could not be sent.",
            )
        ):
            print(text)
    print()
    return result


if __name__ == "__main__":
    main()


### Main program — `main.py`
This is the same terminal entry point used in the local project.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile main.py
"""Main terminal entry point. Run python main.py --help for commands."""

import argparse
import sys
import time
from core import AlertAgent
from demo import main as run_demo


def main(argv=None):
    """Read command-line options and run the requested terminal mode."""
    argument_parser = argparse.ArgumentParser(add_help=False)
    argument_parser.add_argument("--watch", action="store_true")
    argument_parser.add_argument("--interval", type=int, default=900)
    argument_parser.add_argument("--alerts", action="store_true")
    # main.py handles monitoring options. demo.py handles prediction options.
    options, run_arguments = argument_parser.parse_known_args(argv)
    if "--help" in run_arguments or "-h" in run_arguments:
        print(
            "TERMINAL COMMANDS\n  python main.py                       Next three days (requires internet)\n  python main.py --date 2023-08-01     Another historical date\n  python main.py --live               Next three days with current data\n  python main.py --live --watch       Repeat every 15 minutes; Ctrl+C stops\n  python main.py --live --send-email  Email NEW live alerts when configured\n  python main.py --test-alert         Save a local test alert\n  python main.py --send-test-email    Send a TEST email when configured\n  python main.py --alerts             Show saved alerts\n  python train.py                    Train and compare course models\n\nOptional: --interval SECONDS with --watch (minimum 300). No webpage is started."
        )
        return
    # Listing past alerts does not need a new live prediction.
    if options.alerts:
        saved_alerts = AlertAgent().history()
        if not saved_alerts:
            print("No saved alerts.")
        for saved_alert in saved_alerts:
            print(
                f"\n{saved_alert['created_at']} | {saved_alert['title']}\n{saved_alert['body']}"
            )
            print("Email:", saved_alert.get("email", {}).get("status", "not sent"))
        return
    if options.watch and "--live" not in run_arguments:
        raise SystemExit("--watch requires --live.")
    if options.watch and any(
        (x in run_arguments for x in ("--test-alert", "--send-test-email"))
    ):
        raise SystemExit("Run test alerts separately, not repeatedly in watch mode.")
    if options.interval < 300:
        raise SystemExit(
            "Use an interval of at least 300 seconds to avoid excessive public API requests."
        )
    try:
        # Run once normally, or repeat while --watch is enabled.
        while True:
            try:
                run_demo(run_arguments)
            except Exception as error:
                if not options.watch:
                    raise
                print("Update failed:", type(error).__name__, str(error)[:180])
                print("This is not an all-clear. Retrying at the next interval.")
            if not options.watch:
                break
            print(
                f"\nNext check in {options.interval} seconds. Press Ctrl+C to stop.",
                flush=True,
            )
            time.sleep(options.interval)
    # Let the presenter stop monitoring cleanly with Ctrl+C.
    except KeyboardInterrupt:
        print("\nMonitoring stopped.")


if __name__ == "__main__":
    main()


### Train and compare models — `train.py`
Compare Logistic Regression, Multi-Layer Perceptron (MLP), and Support Vector Machine (SVM) with a Radial Basis Function (RBF) kernel.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile train.py
"""Train and compare the wildfire classifiers taught in CS 4150."""

import json
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    precision_score,
    recall_score,
    confusion_matrix,
)
from core import ROOT, features, weather_frame, score


def train_models():
    """Read data, compare classifiers, evaluate, and save the selected model."""
    warnings.filterwarnings("ignore", category=UserWarning)
    # 1. Read actual agency downloads. No synthetic labels are used.
    fire_records = json.loads((ROOT / "data/fires.json").read_text())
    weather_table = weather_frame(
        json.loads((ROOT / "data/weather_history.json").read_text())
    )
    target_dates = pd.date_range("2000-01-15", "2024-12-31")
    input_values = features(target_dates, weather_table)
    fire_dates = {
        pd.to_datetime(fire_record["discovery_date"], unit="ms").normalize()
        for fire_record in fire_records
    }
    # Label 1 means at least one recorded discovery that day; otherwise label 0.
    fire_labels = np.array(
        [int(target_date in fire_dates) for target_date in target_dates]
    )
    # Drop days with missing weather rather than inventing replacement readings.
    has_all_weather = np.isfinite(input_values).all(axis=1)
    input_values = input_values[has_all_weather]
    fire_labels = fire_labels[has_all_weather]
    target_dates = target_dates[has_all_weather]
    # 2. Split by year, keeping later test years separate from learning.
    training_days = target_dates.year <= 2017
    selection_days = (target_dates.year >= 2018) & (target_dates.year <= 2019)
    calibration_days = (target_dates.year >= 2020) & (target_dates.year <= 2021)
    test_days = target_dates.year >= 2022
    # 3. Compare the class methods: sigmoid baseline, neural network, and RBF SVM.
    # The MLP uses sigmoid hidden units, backpropagation, and L2 regularization.
    candidate_models = {
        "Sigmoid logistic baseline": LogisticRegression(max_iter=1000),
        "MLP · 12 → 6 → sigmoid": MLPClassifier(
            hidden_layer_sizes=(12, 6),
            activation="logistic",
            alpha=0.1,
            max_iter=500,
            random_state=4150,
        ),
        "SVM · RBF kernel": SVC(C=1, kernel="rbf", probability=True, random_state=4150),
    }
    trained_models = {}
    comparison_scores = {}
    for model_name, classifier in candidate_models.items():
        # Learn scaling from training rows only to avoid leaking later information.
        trained_model = make_pipeline(StandardScaler(), classifier)
        trained_model.fit(input_values[training_days], fire_labels[training_days])
        comparison_scores[model_name] = float(
            brier_score_loss(
                fire_labels[selection_days],
                trained_model.predict_proba(input_values[selection_days])[:, 1],
            )
        )
        trained_models[model_name] = trained_model
        print(model_name, comparison_scores[model_name], flush=True)
    # 4. Choose using selection years only. Lower Brier score means lower error.
    model_name = min(comparison_scores, key=comparison_scores.get)
    trained_model = trained_models[model_name]
    # 5. Adjust probabilities on separate calibration years, not the final test years.
    probability_adjuster = LogisticRegression(C=1000.0).fit(
        score(trained_model, input_values[calibration_days]),
        fire_labels[calibration_days],
    )
    # 6. Evaluate the chosen, adjusted model on untouched later test dates.
    test_probabilities = probability_adjuster.predict_proba(
        score(trained_model, input_values[test_days])
    )[:, 1]
    training_fire_rate = float(fire_labels[training_days].mean())
    report = {
        "target": "regional recorded discovery day, 50-mile circle",
        "training": "2000–2017",
        "selection": "2018–2019",
        "calibration": "2020–2021",
        "untouched_test": "2022–2024",
        "selection_brier": comparison_scores,
        "selected_model": model_name,
        "records": len(fire_records),
        "samples": int(len(fire_labels)),
        "features": [
            "day-of-year sine",
            "day-of-year cosine",
            "lagged 7-day temperature mean (C)",
            "lagged 7-day humidity mean (%)",
            "lagged 7-day wind mean (m/s)",
            "lagged 7-day precipitation mean (mm/day)",
        ],
        "weather_window": "For target T, average UTC daily observations T-14 through T-8, inclusive; center-point coarse weather only.",
        "test_days": int(test_days.sum()),
        "test_prevalence": float(fire_labels[test_days].mean()),
        "average_precision": float(
            average_precision_score(fire_labels[test_days], test_probabilities)
        ),
        "roc_auc": float(roc_auc_score(fire_labels[test_days], test_probabilities)),
        "brier": float(brier_score_loss(fire_labels[test_days], test_probabilities)),
        "constant_baseline_brier": float(
            brier_score_loss(
                fire_labels[test_days], np.full(test_days.sum(), training_fire_rate)
            )
        ),
        "precision_at_0_5": float(
            precision_score(
                fire_labels[test_days], test_probabilities >= 0.5, zero_division=0
            )
        ),
        "recall_at_0_5": float(
            recall_score(
                fire_labels[test_days], test_probabilities >= 0.5, zero_division=0
            )
        ),
        "confusion_matrix_at_0_5": confusion_matrix(
            fire_labels[test_days], test_probabilities >= 0.5
        ).tolist(),
        "limitations": [
            "Retrospective reanalysis availability is assumed at a conservative 7-day lag; this is not a historical forecast replay.",
            "No fuels, lightning, terrain or human ignition inputs. Center-point weather cannot represent the full 50-mile area.",
            "Discovery dates may differ from ignition dates and records are incomplete. Unrecorded-fire days are treated as negative.",
            "Chronological regional test only; no spatial or prospective operational validation.",
            "Current inference is beyond historical training years. Probability is experimental.",
            "Risk is for any discovery anywhere in the region, not at a specific map point.",
        ],
    }
    # 7. Save the model, its probability adjustment, and the test report together.
    joblib.dump(
        {
            "model": trained_model,
            "calibrator": probability_adjuster,
            "name": model_name,
            "report": report,
        },
        ROOT / "artifacts/model.joblib",
    )
    (ROOT / "artifacts/metrics.json").write_text(json.dumps(report, indent=2))
    (ROOT / "data/Model Data").mkdir(parents=True, exist_ok=True)
    pd.DataFrame(
        {
            "date": target_dates[test_days].strftime("%Y-%m-%d"),
            "observed": fire_labels[test_days],
            "probability": test_probabilities,
        }
    ).to_csv(ROOT / "data/Model Data/test_predictions.csv", index=False)
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    train_models()


### Download fire history again — `download.py`
Optional source downloader. The required data is already bundled, so it is not run automatically.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile download.py
import json
import math
import time
import urllib.request
import urllib.parse
import subprocess
from pathlib import Path

BASE = Path(__file__).resolve().parent
URL = "https://apps.fs.usda.gov/arcx/rest/services/EDW/EDW_FireOccurrenceCurrentEdition_01/MapServer/0/query"
# Ask USFS for reports in a rectangle containing our study area.
query_parameters = {
    "f": "json",
    "where": "fire_year >= 2000",
    "geometry": "-115.0,36.2,-111.1,39.2",
    "geometryType": "esriGeometryEnvelope",
    "inSR": 4326,
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "fod_id,fire_year,discovery_date,discovery_doy,latitude,longitude,fire_size",
    "returnGeometry": "false",
    "orderByFields": "objectid",
    "resultRecordCount": 2000,
}
fire_records = []
# Download every page before applying the exact radius filter.
for offset in range(0, 100000, 2000):
    query_parameters["resultOffset"] = offset
    for attempt in range(3):
        try:
            response_data = json.loads(
                subprocess.check_output(
                    [
                        "curl",
                        "--fail",
                        "-L",
                        "--max-time",
                        "60",
                        "-sS",
                        URL + "?" + urllib.parse.urlencode(query_parameters),
                    ],
                    text=True,
                )
            )
            if "error" in response_data:
                raise RuntimeError(response_data["error"])
            break
        except Exception:
            if attempt == 2:
                raise
            time.sleep(2)
    page_records = [
        fire_record["attributes"] for fire_record in response_data["features"]
    ]
    fire_records += page_records
    print("Downloaded", len(fire_records), flush=True)
    if not response_data.get("exceededTransferLimit"):
        break
else:
    raise RuntimeError("Pagination limit reached")


def distance(latitude, longitude):
    """Calculate miles from Cedar City using latitude and longitude."""
    center_latitude, point_latitude = map(math.radians, [37.6837408, latitude])
    longitude_change = math.radians(longitude + 113.095454)
    return (
        3958.7613
        * 2
        * math.asin(
            math.sqrt(
                math.sin((point_latitude - center_latitude) / 2) ** 2
                + math.cos(center_latitude)
                * math.cos(point_latitude)
                * math.sin(longitude_change / 2) ** 2
            )
        )
    )


# Keep only fire points within 50 miles of Cedar City.
fire_records = [
    fire_record
    for fire_record in fire_records
    if fire_record["latitude"] is not None
    and fire_record["longitude"] is not None
    and (distance(fire_record["latitude"], fire_record["longitude"]) <= 50)
]
(BASE / "data/fires.json").write_text(json.dumps(fire_records))
(BASE / "data/provenance.json").write_text(
    json.dumps(
        {
            "source": URL,
            "edition": "FPA FOD seventh edition, 1992–2024",
            "query": query_parameters,
            "downloaded_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "records": len(fire_records),
            "radius_miles": 50,
            "center": [37.6837408, -113.095454],
        },
        indent=2,
    )
)
print("Within radius:", len(fire_records))


### Export readable CSV files — `export_data.py`
Writes source-named comma-separated values (CSV) files.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile export_data.py
"""Export downloaded official data and model-ready examples as readable CSV files."""

import csv
import json
import numpy as np
import pandas as pd
from core import ROOT, FEATURES, distance, features, weather_frame


def export():
    """Create readable CSV copies of the real downloaded data."""
    for folder in ["US Forest Service", "NASA", "Model Data"]:
        (ROOT / "data" / folder).mkdir(parents=True, exist_ok=True)
    # Preserve original agency data and add readable date/distance columns.
    fire_records = json.loads((ROOT / "data/fires.json").read_text())
    fire_table = pd.DataFrame(fire_records)
    fire_table["discovery_day"] = pd.to_datetime(
        fire_table["discovery_date"], unit="ms"
    ).dt.strftime("%Y-%m-%d")
    fire_table["distance_miles"] = fire_table.apply(
        lambda fire_row: round(distance(fire_row.latitude, fire_row.longitude), 4),
        axis=1,
    )
    fire_table.to_csv(
        ROOT / "data/US Forest Service/fire_history_50_miles.csv", index=False
    )
    weather_table = weather_frame(
        json.loads((ROOT / "data/weather_history.json").read_text())
    )
    weather_table.rename_axis("date_utc").to_csv(
        ROOT / "data/NASA/weather_history.csv", date_format="%Y-%m-%d"
    )
    target_dates = pd.date_range("2000-01-15", "2024-12-31")
    input_values = features(target_dates, weather_table)
    reported_fire_dates = set(
        pd.to_datetime(fire_table["discovery_date"], unit="ms").dt.normalize()
    )
    # One row = one target day. The label says whether a fire was reported.
    training_table = pd.DataFrame(input_values, columns=FEATURES)
    training_table.insert(0, "target_date", target_dates.strftime("%Y-%m-%d"))
    training_table["fire_reported"] = np.array(
        [int(target_date in reported_fire_dates) for target_date in target_dates]
    )
    training_table["split"] = np.select(
        [
            target_dates.year <= 2017,
            target_dates.year <= 2019,
            target_dates.year <= 2021,
        ],
        ["train", "selection", "calibration"],
        default="test",
    )
    training_table = training_table.loc[np.isfinite(input_values).all(axis=1)]
    training_table.to_csv(ROOT / "data/Model Data/training_data.csv", index=False)
    print(
        f"Exported {len(fire_table):,} historical fire records, {len(weather_table):,} weather days, and {len(training_table):,} training examples."
    )
    print(
        "All CSVs are in data/. These are conversions of the downloaded official data, not synthetic data."
    )


if __name__ == "__main__":
    export()


### Email setup helper — `setup_email.py`
Included for completeness. Do not run it in a shared runtime or save credentials in notebook cells.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile setup_email.py
"""Run locally to configure your sender. Password input is hidden."""

import getpass
import os
from pathlib import Path

project_folder = Path(__file__).resolve().parent
print("Cedar Fire email setup → your_email@example.com")
print("Use a Gmail app password for your sending account. Do not paste it into chat.")
sender_email = input("Sending Gmail address: ").strip()
if not sender_email.endswith("@gmail.com") or "\n" in sender_email:
    raise SystemExit("Use a Gmail sender address for this prototype.")
# getpass hides the password while the user types it in their own terminal.
app_password = getpass.getpass("Gmail app password (hidden): ").replace(" ", "")
if len(app_password) != 16 or not app_password.isalnum():
    raise SystemExit(
        "Expected a 16-character Google app password; configuration was not saved."
    )
settings_file = project_folder / ".env"
# Only the current user may read or write the saved password file.
file_handle = os.open(settings_file, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 384)
with os.fdopen(file_handle, "w") as settings_output:
    settings_output.write(
        f"EMAIL_ENABLED=true\nSMTP_HOST=smtp.gmail.com\nSMTP_PORT=465\nSMTP_USER={sender_email}\nSMTP_PASSWORD={app_password}\n"
    )
os.chmod(settings_file, 384)
print("Saved local .env. Run: .venv/bin/python main.py --send-test-email")
print(
    "The terminal prints the sending result; also check your Gmail inbox/spam folder."
)


### Compatibility entry point — `app.py`
Starts the terminal program; there is no webpage.

Edit the code below, then run the cell to save the file in the runtime.

In [ ]:
%%writefile app.py
"""Compatibility entry point: terminal only. No website is started."""

from main import main

if __name__ == "__main__":
    main()


## 3. Train the working model
This creates a model compatible with the current runtime. Training uses 2000–2017; selection uses 2018–2019; calibration uses 2020–2021; final testing uses 2022–2024. Lower Brier score is better. Different library versions can slightly change results.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "train.py"], check=True)


## 4. Show the test results
Precision tells us how often an alert matches a recorded fire day. Recall tells us how many recorded fire days were caught. Neither means the model is always correct.

In [ ]:
import json
import pandas as pd
report = json.loads(Path('artifacts/metrics.json').read_text())
print('Selected model:', report['selected_model'])
print('Test days:', report['test_days'])
print(f"Precision: {report['precision_at_0_5']:.1%}")
print(f"Recall: {report['recall_at_0_5']:.1%}")
print('Confusion matrix:', report['confusion_matrix_at_0_5'])
pd.Series(report['selection_brier'], name='Selection Brier score').sort_values()


## 5. Run a historical example
This works without requesting live weather. It is a historical risk estimate, not a current warning. A high estimate can be a false alarm.

In [ ]:
subprocess.run([sys.executable, "main.py", "--date", "2024-07-01"], check=True)


## 6. Run the next three days
Live requests can fail if a source is unavailable. In that case, use the historical example above. Missing information is not an all-clear. The same daily model is reused; separate accuracy for each future horizon is not established.

In [ ]:
live_run = subprocess.run([sys.executable, 'main.py'], text=True, capture_output=True)
print(live_run.stdout)
if live_run.returncode:
    print('Live data could not be loaded. Use the historical example above.')
    print(live_run.stderr[-1200:])


## 7. Local alert demonstration
This saves a TEST alert inside the runtime. It does not send an email or contact a fire department.

In [ ]:
subprocess.run([sys.executable, "main.py", "--test-alert"], check=True)


## 8. Look at the CSV data
Use the folder button on the left and open `cedar_fire/data` to browse the complete files.

In [ ]:
for filename in sorted(Path('data').rglob('*.csv')):
    table = pd.read_csv(filename)
    print(f'{filename}: {len(table):,} rows')
display(pd.read_csv('data/Model Data/training_data.csv').head())


## 9. Run the project checks

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## Sources and limits
- US Forest Service, Fire Program Analysis Fire-Occurrence Database (FPA FOD), seventh edition: https://doi.org/10.2737/RDS-2013-0009.7
- NASA Prediction Of Worldwide Energy Resources (POWER): https://power.larc.nasa.gov/docs/services/api/temporal/daily/
- National Weather Service (NWS): https://www.weather.gov/documentation/services-web-api
- National Interagency Fire Center (NIFC): https://data-nifc.opendata.arcgis.com/

Weather is from one central point. The model has no fuels, terrain, lightning, or human activity inputs. Discovery dates can differ from ignition dates. The probability concerns the whole region and is experimental. Email requires future setup and testing; fire department notifications are a future step.

To keep your changes, save this notebook in Google Drive. To give teammates access later, use **Share** and grant **Editor** access to their Google accounts.
